# Authors

- David Robredo Manuel
- Duarte Novas Álvarez
- Rubén González Braña

## Imports

In [ ]:
import tensorflow as tf
import keras
import numpy as np
import matplotlib.pyplot as plt
import pandas as pd
import visualkeras

from keras.callbacks import EarlyStopping

from keras.utils import to_categorical

In [ ]:
print("Available Physical Devices: ", tf.config.list_physical_devices())

## Functions

In [ ]:
def show_image(image):

    plt.imshow(image)
    plt.axis("off")
    plt.show()

## Preprocessing

#### LOADING DATA

We load the data and divide it into train and test. Also, we assert that there are the correct number of instances in each group.

In [ ]:
(X_train, y_train), (X_test, y_test) = keras.datasets.cifar100.load_data(label_mode="coarse")
assert X_train.shape == (50000, 32, 32, 3)
assert X_test.shape == (10000, 32, 32, 3)
assert y_train.shape == (50000, 1)
assert y_test.shape == (10000, 1)

Show an example image of the dataset

In [ ]:
show_image(X_train[10])

In [ ]:
y_train = y_train.flatten()
y_test = y_test.flatten()

In [ ]:
X_train = np.array(X_train)
y_train = np.array(y_train)
X_test = np.array(X_test)
y_test = np.array(y_test)

We apply one hot encoding to the target labels

In [ ]:
y_train = to_categorical(y_train, num_classes=20)
y_test = to_categorical(y_test, num_classes=20)

In [ ]:
len(X_train), len(y_train), len(X_test), len(y_test)

### Inception

#### Preprocessing

We apply the proper preprocessing to the data for the inception model

In [ ]:
scaled_X_train = keras.applications.inception_v3.preprocess_input(x = X_train)
scaled_X_test = keras.applications.inception_v3.preprocess_input(x = X_test)

#### Creation

We create an inception model, with an upsampling layer to adjust the size of the images to the size required by the model. We opted for stablishing 'max' pooling

In [ ]:
input_tensor = keras.Input(shape=(32, 32, 3))

x = keras.layers.UpSampling2D(size=(10, 10), interpolation='bilinear')

base_model = keras.applications.InceptionV3(weights='imagenet', include_top=False, input_shape=(320, 320, 3))
base_model.trainable = False

model = keras.models.Sequential([
    input_tensor,
    x,
    base_model,
    keras.layers.GlobalAveragePooling2D(),
    keras.layers.Dense(1024, activation='relu'),
    keras.layers.Dense(20, activation='softmax')  # Cambiar el número de clases a 20
])

In [ ]:
visualkeras.layered_view(model, legend=True)

#### Compilation

In [ ]:
model.compile(optimizer='adam',
              loss='categorical_crossentropy',
              metrics=['accuracy'])

#### Training

In [ ]:
early_stopping_cb = EarlyStopping(patience=15, monitor="val_accuracy", mode="max", restore_best_weights=True)

In [ ]:
history_model = model.fit(
    scaled_X_train,
    y_train,
    validation_split=0.2,
    epochs=200,
    callbacks=[early_stopping_cb]
)

#### Testing

As it can be seen in the training output, this model overfits rapidly to the data, but the regularization methods recover the model that was not yet overfitted. 

In [ ]:
model.evaluate(scaled_X_test, y_test)

In [ ]:
model.save("Model.h5")

In [ ]:
pd.DataFrame(history_model.history).to_csv("Model_Train_History.csv")
pd.DataFrame(history_model.history).plot()

---

### Xception

#### Preprocessing

We apply the proper preprocessing to the data

In [ ]:
scaled_X_train = keras.applications.xception.preprocess_input(x = X_train)
scaled_X_test = keras.applications.xception.preprocess_input(x = X_test)

#### Creation

We upscale the images as we did with previous model

In [ ]:
input_tensor = keras.Input(shape=(32, 32, 3))

x = keras.layers.UpSampling2D(size=(10, 10), interpolation='bilinear')

base_model = keras.applications.Xception(weights='imagenet', include_top=False, input_shape=(320, 320, 3))
base_model.trainable = False

model2 = keras.models.Sequential([
    input_tensor,
    x,
    base_model,
    keras.layers.GlobalAveragePooling2D(),
    keras.layers.Dense(1024, activation='relu'),
    keras.layers.Dense(20, activation='softmax')  # Cambiar el número de clases a 20
])

In [ ]:
visualkeras.layered_view(model2, legend=True)

#### Compilation

In [ ]:
model2.compile(optimizer='adam',
              loss='categorical_crossentropy',
              metrics=['accuracy'])

#### Training

In [ ]:
early_stopping_cb = EarlyStopping(patience=15, monitor="val_loss", mode="min", restore_best_weights=True)

In [ ]:
history_model2 = model2.fit(
    scaled_X_train,
    y_train,
    validation_split=0.2,
    epochs=200,
    callbacks=[early_stopping_cb]
)

#### Testing

With the Xception model, we obtain our best results, thanks to the model itself, but also to the preprocessing and the regularization methods

In [ ]:
model2.evaluate(scaled_X_test, y_test)

In [ ]:
model2.save("Model2.h5")

In [ ]:
pd.DataFrame(history_model2.history).to_csv("Model2_Train_History.csv")
pd.DataFrame(history_model2.history).plot()

---

### Residual

#### Preprocessing

We apply again the correct preprocessing

In [ ]:
scaled_X_train = keras.applications.resnet50.preprocess_input(x = X_train)
scaled_X_test = keras.applications.resnet50.preprocess_input(x = X_test)

#### Creation

In [ ]:
input_tensor = keras.Input(shape=(32, 32, 3))

x = keras.layers.UpSampling2D(size=(10, 10), interpolation='bilinear')

base_model = keras.applications.ResNet50(weights='imagenet', include_top=False, input_shape=(320, 320, 3))
base_model.trainable = False

model3 = keras.models.Sequential([
    input_tensor,
    x,
    base_model,
    keras.layers.GlobalAveragePooling2D(),
    keras.layers.Dense(1024, activation='relu'),
    keras.layers.Dense(20, activation='softmax')  # Cambiar el número de clases a 20
])

In [ ]:
visualkeras.layered_view(model3, legend=True)

#### Compilation

In [ ]:
model3.compile(optimizer='adam',
                loss='categorical_crossentropy',
                metrics=['accuracy'])

#### Training

In [ ]:
early_stopping_cb = EarlyStopping(patience=15, monitor='val_loss', mode='min', restore_best_weights=True)

In [ ]:
history_model3 = model3.fit(
    scaled_X_train,
    y_train,
    validation_split=0.2,
    epochs=200,
    callbacks=[early_stopping_cb]
)

#### Testing

The Residual model obtains better results than preivous models except from the Xception model

In [ ]:
model3.evaluate(scaled_X_test, y_test)

In [ ]:
model3.save('Model7.h5')

In [ ]:
pd.DataFrame(history_model3.history).to_csv("Model7_Train_History.csv")
pd.DataFrame(history_model3.history).plot()